# FINAL CLEANED METADATA PIPELINE — Emotion Detection System

**Corrected complete version**

This notebook fixes the confirmed GoEmotions metadata-loss bug.

### Critical correction
The GoEmotions parser decoded `text` correctly, but the previous `build_dataset_metadata()` branch discarded that field when constructing `emotion_metadata`. This version preserves the actual text payload through:

`raw CSV → decoder → emotion_metadata → global dedup → final training manifest`

### Execution order
Run sequentially from Cell 1 through Cell 8. The final manifest remains `UNASSIGNED` for train/validation/test; splitting is handled by the dedicated split notebook.

**Do not use the previous notebook's Cell 7 or Cell 8.**


In [1]:
# CELL 1 — IMPORTS
# =============================================================

import os
import glob
import json
import hashlib
import wave
import ast
import re
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
from PIL import Image

warnings.filterwarnings("ignore")

print("✓ Imports loaded.")

# =============================================================

✓ Imports loaded.


In [2]:
# CELL 2 — FINAL PROJECT CONFIGURATION
# =============================================================

PROJECT_DIR = Path(r"C:\New folder\New Emodect")
CONFIG_DIR = PROJECT_DIR / "config"
REPORTS_DIR = PROJECT_DIR / "reports"
CLEANED_DIR = PROJECT_DIR / "cleaned_metadata"

CLEANED_DIR.mkdir(parents=True, exist_ok=True)

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Project directory not found: {PROJECT_DIR}")

CANONICAL_EMOTIONS = [
    "anger", "disgust", "fear", "happiness",
    "sadness", "surprise", "neutral"
]
CANONICAL_EMOTION_SET = set(CANONICAL_EMOTIONS)

RAVDESS_CALM_TO_NEUTRAL = True
IEMOCAP_EXC_TO_HAPPINESS = True
EXCLUDE_SAMM_FROM_STATIC = True
INCLUDE_GOEMOTIONS = True

DATASET_PATHS = {
    "CREMA-D": {
        "path": PROJECT_DIR / "datasets" / "AudioWAV",
        "type": "audio", "format": ["**/*.wav"],
    },
    "RAVDESS": {
        "path": PROJECT_DIR / "datasets" / "RAVDESS_Dataset",
        "type": "mixed", "format": ["**/*.wav", "**/*.mp4"],
    },
    "SAVEE": {
        "path": PROJECT_DIR / "datasets" / "ejlok1" / "surrey-audiovisual-expressed-emotion-savee",
        "type": "audio", "format": ["**/*.wav"],
    },
    "TESS": {
        "path": PROJECT_DIR / "datasets" / "ejlok1" / "toronto-emotional-speech-set-tess" /
               "versions" / "1" / "TESS Toronto emotional speech set data" /
               "TESS Toronto emotional speech set data",
        "type": "audio", "format": ["**/*.wav"],
    },
    "IEMOCAP": {
        "path": PROJECT_DIR / "datasets" / "dejolilandry" / "iemocapfullrelease" /
               "versions" / "1" / "IEMOCAP_full_release",
        "type": "mixed", "format": ["**/*.wav", "**/*.mp4"],
    },
    "FER2013": {
        "path": PROJECT_DIR / "datasets" / "msambare" / "fer2013",
        "type": "image", "format": ["**/*.jpg", "**/*.png"],
    },
    "AffectNet": {
        "path": PROJECT_DIR / "datasets" / "mstjebashazida" / "affectnet",
        "type": "image", "format": ["**/*.jpg", "**/*.png"],
    },
    "CK+": {
        "path": PROJECT_DIR / "datasets" / "shawon10" / "ckplus",
        "type": "image", "format": ["**/*.jpg", "**/*.png"],
    },
    "RAF-DB": {
        "path": PROJECT_DIR / "datasets" / "shuvoalok" / "raf-db-dataset",
        "type": "image", "format": ["**/*.jpg", "**/*.png"],
    },
    "GoEmotions": {
        "path": PROJECT_DIR / "datasets" / "debarshichanda" / "goemotions" /
               "versions" / "6" / "data" / "full_dataset",
        "type": "text", "format": ["**/*.csv"],
    },
    "SAMM": {
        "path": PROJECT_DIR / "datasets" / "SAMM",
        "type": "image", "format": ["**/*.jpg", "**/*.png"],
    },
}

print("✓ Project:", PROJECT_DIR)
print("✓ Canonical classes:", CANONICAL_EMOTIONS)
print("✓ GoEmotions ACTIVE:", INCLUDE_GOEMOTIONS)

# =============================================================

✓ Project: C:\New folder\New Emodect
✓ Canonical classes: ['anger', 'disgust', 'fear', 'happiness', 'sadness', 'surprise', 'neutral']
✓ GoEmotions ACTIVE: True


In [3]:
# CELL 3 — FINAL SEVEN-CLASS LABEL MAP
# =============================================================

DATASET_LABEL_MAP = {
    "CREMA-D": {
        "ANG": "anger", "DIS": "disgust", "FEA": "fear",
        "HAP": "happiness", "SAD": "sadness", "SUR": "surprise",
        "NEU": "neutral",
    },
    "RAVDESS": {
        "01": "neutral", "02": "neutral" if RAVDESS_CALM_TO_NEUTRAL else None,
        "03": "happiness", "04": "sadness", "05": "anger",
        "06": "fear", "07": "disgust", "08": "surprise",
    },
    "SAVEE": {
        "A": "anger", "D": "disgust", "F": "fear",
        "H": "happiness", "N": "neutral", "SA": "sadness",
        "SU": "surprise",
    },
    "TESS": {
        "ANGER": "anger", "ANGRY": "anger",
        "DISGUST": "disgust", "FEAR": "fear",
        "HAPPY": "happiness", "HAPPINESS": "happiness",
        "SAD": "sadness", "SADNESS": "sadness",
        "SURPRISE": "surprise", "PLEASANT_SURPRISE": "surprise",
        "PLEASANT_SURPRISED": "surprise",
        "NEUTRAL": "neutral", "CALM": "neutral",
    },
    "IEMOCAP": {
        "ANG": "anger", "DIS": "disgust", "FEA": "fear",
        "HAP": "happiness", "EXC": "happiness" if IEMOCAP_EXC_TO_HAPPINESS else None,
        "SAD": "sadness", "SUR": "surprise", "NEU": "neutral",
    },
    "FER2013": {
        "0": "anger", "1": "disgust", "2": "fear",
        "3": "happiness", "4": "sadness", "5": "surprise", "6": "neutral",
        "ANGER": "anger", "ANGRY": "anger",
        "DISGUST": "disgust", "DISGUSTED": "disgust",
        "FEAR": "fear", "FEARFUL": "fear",
        "HAPPY": "happiness", "HAPPINESS": "happiness",
        "SAD": "sadness", "SADNESS": "sadness",
        "SURPRISE": "surprise", "SURPRISED": "surprise",
        "NEUTRAL": "neutral",
    },
    "AffectNet": {
        "ANGER": "anger", "DISGUST": "disgust", "FEAR": "fear",
        "HAPPINESS": "happiness", "HAPPY": "happiness",
        "SADNESS": "sadness", "SAD": "sadness",
        "SURPRISE": "surprise", "NEUTRAL": "neutral",
    },
    "CK+": {
        "ANGER": "anger", "DISGUST": "disgust", "FEAR": "fear",
        "HAPPINESS": "happiness", "HAPPY": "happiness",
        "SADNESS": "sadness", "SAD": "sadness",
        "SURPRISE": "surprise", "NEUTRAL": "neutral",
    },
    "RAF-DB": {
        "1": "surprise", "2": "fear", "3": "disgust", "4": "happiness",
        "5": "sadness", "6": "anger", "7": "neutral",
    },
    "GoEmotions": {
        # Conservative harmonization: only source emotions with a
        # defensible correspondence to the seven-class target.
        "AMUSEMENT": "happiness",
        "EXCITEMENT": "happiness",
        "JOY": "happiness",
        "LOVE": "happiness",
        "ANGER": "anger",
        "ANNOYANCE": "anger",
        "DISAPPROVAL": "anger",
        "SADNESS": "sadness",
        "DISAPPOINTMENT": "sadness",
        "GRIEF": "sadness",
        "REMORSE": "sadness",
        "FEAR": "fear",
        "NERVOUSNESS": "fear",
        "DISGUST": "disgust",
        "SURPRISE": "surprise",
        "NEUTRAL": "neutral",
    },
    "SAMM": {},
}

def normalize_label(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    value = str(value).strip()
    return value.upper() if value else None

def map_label(dataset, raw_label):
    return DATASET_LABEL_MAP.get(dataset, {}).get(normalize_label(raw_label))

for dataset, mapping in DATASET_LABEL_MAP.items():
    for raw, canonical in mapping.items():
        if canonical is not None and canonical not in CANONICAL_EMOTION_SET:
            raise ValueError(f"Invalid mapping: {dataset}/{raw}->{canonical}")

print("✓ Seven-class label map validated.")

# =============================================================

✓ Seven-class label map validated.


In [4]:
# CELL 4 — DISCOVERY, INTEGRITY, HASHING, AND VERIFICATION CONTEXT
# =============================================================

def discover_files(dataset_path, patterns):
    found = set()

    for pattern in patterns:
        found.update(
            glob.glob(
                os.path.join(str(dataset_path), pattern),
                recursive=True
            )
        )

    return sorted(found)


def md5_file(filepath, chunk_size=1024 * 1024):
    hasher = hashlib.md5()

    try:
        with open(filepath, "rb") as f:
            while True:
                chunk = f.read(chunk_size)

                if not chunk:
                    break

                hasher.update(chunk)

        return hasher.hexdigest()

    except Exception:
        return None


def validate_file(filepath, dataset_type):
    """
    Current filesystem integrity is authoritative.

    This function deliberately returns (valid, reason), so a Python
    exception can never be misreported as a corrupted file.
    """

    try:
        if not os.path.exists(filepath):
            return False, "FILE_NOT_FOUND"

        if os.path.getsize(filepath) <= 0:
            return False, "ZERO_BYTE_FILE"

        ext = Path(filepath).suffix.lower()


        # -----------------------------------------------------
        # WAV
        # -----------------------------------------------------

        if ext == ".wav":

            with wave.open(filepath, "rb") as wav:

                if wav.getnframes() <= 0:
                    return False, "EMPTY_WAV"

                if wav.getnchannels() <= 0:
                    return False, "INVALID_CHANNEL_COUNT"

                if wav.getframerate() <= 0:
                    return False, "INVALID_SAMPLE_RATE"

            return True, "VALID"


        # -----------------------------------------------------
        # IMAGE
        # -----------------------------------------------------

        if ext in {
            ".jpg",
            ".jpeg",
            ".png",
            ".bmp",
            ".tif",
            ".tiff"
        }:

            with Image.open(filepath) as img:
                img.verify()

            return True, "VALID"


        # -----------------------------------------------------
        # CSV
        # -----------------------------------------------------

        if ext == ".csv":

            pd.read_csv(
                filepath,
                nrows=5
            )

            return True, "VALID"


        # -----------------------------------------------------
        # JSON
        # -----------------------------------------------------

        if ext == ".json":

            try:

                with open(
                    filepath,
                    "r",
                    encoding="utf-8"
                ) as f:

                    json.load(f)

            except json.JSONDecodeError:

                pd.read_json(
                    filepath,
                    lines=True,
                    nrows=5
                )

            return True, "VALID"


        # -----------------------------------------------------
        # MP4
        # -----------------------------------------------------
        #
        # Size/existence only at metadata stage.
        # Full video decoding belongs to preprocessing.
        # -----------------------------------------------------

        if ext == ".mp4":
            return True, "VALID_SIZE_ONLY"


        # -----------------------------------------------------
        # Generic
        # -----------------------------------------------------

        return True, "VALID_SIZE_ONLY"


    except Exception as e:

        return False, f"INTEGRITY_EXCEPTION:{repr(e)}"


def load_verification_context():
    """
    Reads the final verification summary/log for provenance.

    The verification report is contextual evidence only.
    Current filesystem validation above remains authoritative so
    repaired files are not permanently excluded by a stale report.
    """

    context = {
        "summary_available": False,
        "summary": None,
        "corrupted_from_report": {},
    }


    summary_path = (
        REPORTS_DIR /
        "verification_summary.csv"
    )

    log_path = (
        REPORTS_DIR /
        "detailed_verification_log.csv"
    )


    if summary_path.exists():

        try:

            context["summary"] = pd.read_csv(
                summary_path
            )

            context["summary_available"] = True

        except Exception:
            pass


    if log_path.exists():

        try:

            log = pd.read_csv(
                log_path
            )

            required = {
                "Dataset",
                "File",
                "Issue"
            }

            if required.issubset(log.columns):

                for _, row in log.iterrows():

                    issue = str(
                        row["Issue"]
                    ).strip().lower()

                    if "corrupt" not in issue:
                        continue

                    dataset = str(
                        row["Dataset"]
                    ).strip()

                    filename = os.path.basename(
                        str(row["File"]).strip()
                    )

                    if (
                        filename
                        and
                        filename.lower() != "nan"
                    ):

                        context[
                            "corrupted_from_report"
                        ].setdefault(
                            dataset,
                            set()
                        ).add(
                            filename
                        )

        except Exception:
            pass


    return context


VERIFICATION_CONTEXT = (
    load_verification_context()
)


print(
    "✓ Verification context loaded:",
    VERIFICATION_CONTEXT["summary_available"]
)

# =============================================================

✓ Verification context loaded: True


In [5]:
# CELL 5 — FINAL DATASET-SPECIFIC PARSERS
# =============================================================

def parse_cremad(file_paths):
    result = {}

    for path in file_paths:

        stem = Path(path).stem
        parts = stem.split("_")

        if len(parts) >= 3:
            result[path] = parts[2].upper()

    return result


def parse_ravdess(file_paths):
    result = {}

    for path in file_paths:

        parts = Path(path).stem.split("-")

        if len(parts) == 7:
            result[path] = parts[2]

    return result


def parse_savee(file_paths):
    """
    Correct SAVEe parsing.

    Examples:
        DC_A_001  -> A
        JE_SA_012 -> SA
        KL_SU_123 -> SU
    """

    result = {}

    valid_codes = {
        "A",
        "D",
        "F",
        "H",
        "N",
        "SA",
        "SU",
    }


    for path in file_paths:

        stem = Path(path).stem.upper()

        # Remove the trailing utterance number from the
        # entire filename, not only from the last token.
        stem_without_number = re.sub(
            r"\d+$",
            "",
            stem
        ).rstrip("_")


        parts = stem_without_number.split("_")


        if len(parts) >= 2:

            emotion_code = parts[-1]

            if emotion_code in valid_codes:

                result[path] = emotion_code


    return result


def parse_tess(file_paths):
    result = {}

    for path in file_paths:

        parent = Path(path).parent.name

        if "_" not in parent:
            continue

        label = parent.split(
            "_",
            1
        )[1].strip()


        if label:
            result[path] = label.upper()


    return result


def parse_iemocap(file_paths, dataset_path):

    labels = {}


    annotation_files = glob.glob(

        os.path.join(
            str(dataset_path),
            "**",
            "EmoEvaluation",
            "*.txt"
        ),

        recursive=True

    )


    for annotation_file in annotation_files:

        try:

            with open(
                annotation_file,
                "r",
                encoding="utf-8",
                errors="ignore"
            ) as f:

                for line in f:

                    if not line.startswith("["):
                        continue


                    parts = (
                        line.rstrip("\n")
                        .split("\t")
                    )


                    if len(parts) < 3:
                        continue


                    sample_id = parts[1].strip()
                    label = parts[2].strip()


                    if sample_id and label:

                        labels[
                            sample_id
                        ] = label.upper()


        except Exception:
            continue


    result = {}


    for path in file_paths:

        stem = Path(path).stem

        if stem in labels:

            result[path] = labels[stem]


    return result


def parse_folder_based(file_paths):
    """
    Robust folder-label parser.

    Some datasets (notably FER2013) place images below one or more
    intermediate folders. Therefore the parser searches the full
    ancestor chain rather than assuming the immediate parent is the
    emotion label.
    """
    result = {}
    for path in file_paths:
        parts = [str(x).strip() for x in Path(path).parts[::-1]]
        # Preserve the first meaningful folder encountered from the file upward.
        for part in parts[1:]:
            token = part.strip().upper()
            if token:
                result[path] = token
                break
    return result

def parse_fer2013(file_paths):
    """
    FER2013 supports both numeric directory names (0-6) and common
    human-readable directory names. Search ancestors so layouts such as
    train/happy/file.jpg and train/3/file.jpg are both handled.
    """
    aliases = {
        "0":"0", "ANGER":"0", "ANGRY":"0",
        "1":"1", "DISGUST":"1", "DISGUSTED":"1",
        "2":"2", "FEAR":"2", "FEARFUL":"2",
        "3":"3", "HAPPY":"3", "HAPPINESS":"3",
        "4":"4", "SAD":"4", "SADNESS":"4",
        "5":"5", "SURPRISE":"5", "SURPRISED":"5",
        "6":"6", "NEUTRAL":"6",
    }
    result = {}
    for path in file_paths:
        for part in Path(path).parts[::-1]:
            token = str(part).strip().upper()
            if token in aliases:
                result[path] = aliases[token]
                break
    return result


def parse_rafdb(file_paths, dataset_path):

    labels = {}


    label_file = (
        Path(dataset_path)
        / "EmoLabel"
        / "list_patition_label.txt"
    )


    if label_file.exists():

        with open(
            label_file,
            "r",
            encoding="utf-8",
            errors="ignore"
        ) as f:

            for line in f:

                parts = line.split()

                if len(parts) >= 2:

                    key = (
                        Path(parts[0]).stem
                        .replace("_aligned", "")
                    )

                    labels[key] = str(parts[1])


    if not labels:

        label_csvs = glob.glob(

            os.path.join(
                str(dataset_path),
                "**",
                "*labels.csv"
            ),

            recursive=True

        )


        frames = []


        for csv_path in label_csvs:

            try:
                frames.append(
                    pd.read_csv(csv_path)
                )

            except Exception:
                continue


        if frames:

            df = pd.concat(
                frames,
                ignore_index=True
            )


            if {
                "image",
                "label"
            }.issubset(df.columns):

                for _, row in df.iterrows():

                    key = (
                        Path(
                            str(row["image"])
                        ).stem
                        .replace(
                            "_aligned",
                            ""
                        )
                    )

                    labels[key] = str(
                        row["label"]
                    )


    result = {}


    for path in file_paths:

        key = (
            Path(path).stem
            .replace("_aligned", "")
        )


        if key in labels:

            result[path] = labels[key].upper()


    return result


def parse_samm(file_paths, dataset_path):
    """
    SAMM is sequence/micro-expression data.

    Its annotation structure is not converted into fabricated static
    frame-level labels in this pipeline.
    """

    return {}


PARSERS = {

    "CREMA-D":
        lambda paths, root: parse_cremad(paths),

    "RAVDESS":
        lambda paths, root: parse_ravdess(paths),

    "SAVEE":
        lambda paths, root: parse_savee(paths),

    "TESS":
        lambda paths, root: parse_tess(paths),

    "IEMOCAP":
        lambda paths, root: parse_iemocap(paths, root),

    "FER2013":
        lambda paths, root: parse_fer2013(paths),

    "AffectNet":
        lambda paths, root: parse_folder_based(paths),

    "CK+":
        lambda paths, root: parse_folder_based(paths),

    "RAF-DB":
        lambda paths, root: parse_rafdb(paths, root),

}


print(
    "✓ Dataset parsers registered:",
    len(PARSERS)
)

assert set(PARSERS) == {
    "CREMA-D",
    "RAVDESS",
    "SAVEE",
    "TESS",
    "IEMOCAP",
    "FER2013",
    "AffectNet",
    "CK+",
    "RAF-DB",
}

print("✓ Parser registry validated.")

✓ Dataset parsers registered: 9
✓ Parser registry validated.


In [6]:
# ============================================================
# GOEMOTIONS: AUTHORITATIVE ONE-HOT DECODER
# ============================================================
# The installed GoEmotions source is CSV-based and contains 28
# one-hot emotion columns. There is intentionally NO single
# "label"/"emotion" column assumption here.
#
# Policy:
#   1. Decode every active one-hot GoEmotions emotion column.
#   2. Map source emotions to the project's seven canonical classes.
#   3. If all mapped canonical candidates collapse to ONE class,
#      accept the row (even if unsupported source labels co-occur).
#   4. If multiple canonical classes are present, exclude as ambiguous.
#   5. If no canonical class is present, exclude as unsupported-only.
#   6. Empty/invalid text is excluded.
#   7. Rows marked example_very_unclear are excluded by default.

GOEMOTIONS_EXPECTED_LABELS = [
    "admiration", "amusement", "anger", "annoyance", "approval",
    "caring", "confusion", "curiosity", "desire", "disappointment",
    "disapproval", "disgust", "embarrassment", "excitement", "fear",
    "gratitude", "grief", "joy", "love", "nervousness", "optimism",
    "pride", "realization", "relief", "remorse", "sadness",
    "surprise", "neutral"
]

GOEMOTIONS_MAP = {
    "amusement": "happiness",
    "excitement": "happiness",
    "joy": "happiness",
    "love": "happiness",

    "anger": "anger",
    "annoyance": "anger",
    "disapproval": "anger",

    "sadness": "sadness",
    "disappointment": "sadness",
    "grief": "sadness",
    "remorse": "sadness",

    "fear": "fear",
    "nervousness": "fear",

    "disgust": "disgust",
    "surprise": "surprise",
    "neutral": "neutral",
}

GOEMOTIONS_CANONICAL = set(GOEMOTIONS_MAP.values())
GOEMOTIONS_EXCLUDE_VERY_UNCLEAR = True

def _norm_col(x):
    return str(x).strip().lower()

def detect_goemotions_onehot_columns(df):
    """Return the exact 28 GoEmotions one-hot columns, case-insensitively."""
    by_norm = {_norm_col(c): c for c in df.columns}
    missing = [x for x in GOEMOTIONS_EXPECTED_LABELS if x not in by_norm]
    if missing:
        raise ValueError(
            "GoEmotions schema error: missing one-hot emotion columns: "
            + repr(missing)
        )
    return [by_norm[x] for x in GOEMOTIONS_EXPECTED_LABELS]

def decode_goemotions_row(row, onehot_cols):
    """
    Decode one row from the real GoEmotions one-hot schema.

    Returns:
        (canonical_label_or_None, reason, active_source_labels, canonical_candidates)
    """
    text = row.get("text")
    if text is None or not isinstance(text, str) or not text.strip():
        return None, "missing_or_empty_text", [], []

    if GOEMOTIONS_EXCLUDE_VERY_UNCLEAR:
        unclear = row.get("example_very_unclear", False)
        if bool(unclear):
            return None, "example_very_unclear", [], []

    active = []
    for col in onehot_cols:
        try:
            value = int(row[col])
        except Exception:
            value = 0
        if value == 1:
            active.append(_norm_col(col))

    if not active:
        return None, "missing_or_unparsed_labels", [], []

    canonical = sorted(set(
        GOEMOTIONS_MAP[label]
        for label in active
        if label in GOEMOTIONS_MAP
    ))

    if len(canonical) == 1:
        return canonical[0], "valid_one_canonical_class", active, canonical
    if len(canonical) > 1:
        return None, "ambiguous_multiple_canonical_classes", active, canonical
    return None, "unsupported_only_labels", active, canonical

In [7]:
# ============================================================
# GOEMOTIONS ROW PARSER
# ============================================================
def parse_goemotions_rows(file_paths):
    """
    Parse the real GoEmotions CSV layout.

    Each CSV contains:
      text + metadata + 28 one-hot emotion columns +
      example_very_unclear.

    Returns:
      parsed_df:
        one row per raw source row with canonical/QA status.
      audit_df:
        compact accounting by reason and canonical emotion.
    """
    parsed_rows = []
    reason_counter = Counter()
    canonical_counter = Counter()

    for csv_path in sorted(file_paths):
        df = pd.read_csv(csv_path)

        onehot_cols = detect_goemotions_onehot_columns(df)

        if "text" not in df.columns:
            raise ValueError(f"GoEmotions file has no text column: {csv_path}")

        for row_index, (_, row) in enumerate(df.iterrows()):
            canonical, reason, active_labels, canonical_candidates = \
                decode_goemotions_row(row, onehot_cols)

            text = row.get("text")
            text_value = "" if text is None else str(text)

            if active_labels:
                original_label = "|".join(active_labels)
            else:
                original_label = ""

            parsed_rows.append({
                "file_path": str(Path(csv_path).resolve()),
                "row_index": int(row_index),
                "source_id": str(row.get("id", "")),
                "text": text_value,
                "original_label": original_label,
                "mapped_emotion": canonical,
                "status": "clean" if canonical is not None else "excluded",
                "reason": reason,
                "active_source_labels": "|".join(active_labels),
                "canonical_candidates": "|".join(canonical_candidates),
            })

            reason_counter[reason] += 1
            if canonical is not None:
                canonical_counter[canonical] += 1

    parsed_df = pd.DataFrame(parsed_rows)

    audit_rows = []
    for reason, count in sorted(reason_counter.items()):
        audit_rows.append({
            "category": "reason",
            "label": reason,
            "count": int(count),
        })

    for label, count in sorted(canonical_counter.items()):
        audit_rows.append({
            "category": "canonical_emotion",
            "label": label,
            "count": int(count),
        })

    audit_df = pd.DataFrame(audit_rows)

    return parsed_df, audit_df

print("✓ GoEmotions row parser defined.")

# =============================================================

✓ GoEmotions row parser defined.


In [8]:
# CELL 7 — BUILD FINAL SEVEN-CLASS METADATA
# =============================================================

def build_sample_id(dataset, file_path, row_index=None):
    raw = f"{dataset}|{os.path.normcase(os.path.abspath(str(file_path)))}"
    if row_index is not None:
        raw += f"|row={row_index}"
    return hashlib.sha1(raw.encode("utf-8")).hexdigest()

def infer_modality(dataset, file_path, config):
    if dataset == "GoEmotions":
        return "text"
    ext = Path(file_path).suffix.lower()
    if ext == ".wav": return "audio"
    if ext == ".mp4": return "video"
    if ext in {".jpg",".jpeg",".png",".bmp",".tif",".tiff"}: return "image"
    return config["type"]

def infer_group_id(dataset, file_path, row_index=None):
    p = Path(file_path)
    stem = p.stem
    if dataset == "GoEmotions":
        # GoEmotions has no defensible speaker identity in this source.
        # Each text row is therefore a sample-level group.
        return f"GoEmotions|sample|{build_sample_id(dataset, file_path, row_index)}"
    if dataset == "CREMA-D":
        parts = stem.split("_")
        return f"CREMA-D|speaker|{parts[0]}" if parts and parts[0] else build_sample_id(dataset,file_path)
    if dataset == "RAVDESS":
        parts = stem.split("-")
        return f"RAVDESS|actor|{parts[6]}" if len(parts)==7 and parts[6].isdigit() else build_sample_id(dataset,file_path)
    if dataset == "IEMOCAP":
        for part in p.parts:
            if str(part).lower().startswith("session"):
                return f"IEMOCAP|{str(part).lower()}"
    if dataset == "SAVEE":
        prefix = stem[:2].upper()
        return f"SAVEE|speaker|{prefix}" if prefix else build_sample_id(dataset,file_path)
    if dataset == "TESS":
        parent = p.parent.name
        prefix = parent.split("_",1)[0].upper() if "_" in parent else ""
        return f"TESS|speaker|{prefix}" if prefix else build_sample_id(dataset,file_path)
    return f"{dataset}|sample|{build_sample_id(dataset,file_path)}"

def build_dataset_metadata(dataset_name, config):
    root = config["path"]
    if not root.exists():
        return pd.DataFrame([{
            "sample_id":"","dataset":dataset_name,"task":"emotion",
            "modality":config["type"],"file_path":str(root),
            "original_label":None,"mapped_emotion":None,
            "status":"excluded","reason":"dataset_path_missing",
            "group_id":"","group_source":"none","duplicate_hash":"",
            "duplicate_group_size":0,"is_exact_duplicate":False,
            "text":"",
        }])

    if dataset_name == "SAMM" and EXCLUDE_SAMM_FROM_STATIC:
        return pd.DataFrame([{
            "sample_id":"","dataset":"SAMM","task":"emotion","modality":"image",
            "file_path":str(root),"original_label":None,"mapped_emotion":None,
            "status":"excluded","reason":"samm_temporal_dataset_not_used_in_static_pipeline",
            "group_id":"","group_source":"none","duplicate_hash":"",
            "duplicate_group_size":0,"is_exact_duplicate":False,
            "text":"",
        }])

    paths = discover_files(root, config["format"])

    if dataset_name == "GoEmotions":
        go_df, audit = parse_goemotions_rows(paths)
        if go_df.empty:
            raise RuntimeError("GoEmotions was found but produced zero parsed rows.")
        rows = []
        for _, r in go_df.iterrows():
            fp = r["file_path"]; ri = int(r["row_index"])
            sample_id = build_sample_id(dataset_name, fp, ri)
            text = str(r.get("text",""))
            digest = hashlib.sha256(re.sub(r"\s+"," ",text).strip().lower().encode("utf-8")).hexdigest() if text.strip() else ""
            rows.append({
                "sample_id":sample_id,"dataset":"GoEmotions","task":"emotion",
                "modality":"text","file_path":fp,
                "original_label":r["original_label"],"mapped_emotion":r["mapped_emotion"],
                "status":r["status"],"reason":r["reason"],
                "group_id":infer_group_id(dataset_name,fp,ri),
                "group_source":"sample_only_subject_identity_unavailable",
                "duplicate_hash":digest,"duplicate_group_size":0,
                "is_exact_duplicate":False,
                # CRITICAL FIX: preserve actual GoEmotions text.
                "text":text,
            })
        out = pd.DataFrame(rows)
        counts = out["duplicate_hash"].value_counts()
        out["duplicate_group_size"] = out["duplicate_hash"].map(counts).fillna(0).astype(int)
        out["is_exact_duplicate"] = out["duplicate_hash"].ne("") & out["duplicate_group_size"].gt(1)
        audit.to_csv(CLEANED_DIR/"GoEmotions_parsing_audit.csv", index=False)
        print("\nGoEmotions parsing audit:")
        display(audit)
        return out

    parser = PARSERS.get(dataset_name)
    if parser is None:
        raise ValueError(f"No parser registered for {dataset_name}")

    parsed = parser(paths, root)
    integrity, hashes = {}, {}
    for path in paths:
        ok, reason = validate_file(path, config["type"])
        integrity[path] = (ok, reason)
        hashes[path] = md5_file(path) if ok else ""

    hash_to_paths = defaultdict(list)
    for path, digest in hashes.items():
        if digest:
            hash_to_paths[digest].append(path)

    rows = []
    identity_grouped = {"CREMA-D","RAVDESS","IEMOCAP","SAVEE","TESS"}
    for path in paths:
        ok, integrity_reason = integrity[path]
        digest = hashes[path]
        base = {
            "sample_id":build_sample_id(dataset_name,path),
            "dataset":dataset_name,"task":"emotion",
            "modality":infer_modality(dataset_name,path,config),
            "file_path":str(Path(path).resolve()),
            "original_label":None,"mapped_emotion":None,
            "status":"excluded","reason":"",
            "group_id":infer_group_id(dataset_name,path),
            "group_source":"dataset_encoded_identity_or_session" if dataset_name in identity_grouped else "sample_only_subject_identity_unavailable",
            "duplicate_hash":digest,
            "duplicate_group_size":len(hash_to_paths.get(digest,[])),
            "is_exact_duplicate":bool(digest and len(hash_to_paths.get(digest,[]))>1),
        }
        if not ok:
            base["reason"] = f"integrity_failure:{integrity_reason}"
            rows.append(base); continue
        raw_label = parsed.get(path)
        if raw_label is None:
            base["reason"]="missing_annotation"
            rows.append(base); continue
        base["original_label"]=raw_label
        canonical=map_label(dataset_name,raw_label)
        base["mapped_emotion"]=canonical
        if canonical is None:
            base["reason"]="unsupported_emotion"
        elif canonical not in CANONICAL_EMOTION_SET:
            base["reason"]="invalid_canonical_label"
        else:
            base["status"]="clean"
        rows.append(base)
    return pd.DataFrame(rows)

all_metadata=[]
for dataset_name, config in DATASET_PATHS.items():
    print(f"Processing {dataset_name}...")
    df=build_dataset_metadata(dataset_name,config)
    path=CLEANED_DIR/f"{dataset_name}_metadata.csv"
    df.to_csv(path,index=False)
    all_metadata.append(df)
    print(f"  ✓ rows={len(df):,} clean={(df.status=='clean').sum():,} excluded={(df.status=='excluded').sum():,}")

emotion_metadata=pd.concat(all_metadata,ignore_index=True)

# HARD METADATA TEXT GATE
assert "text" in emotion_metadata.columns, (
    "METADATA GATE FAIL — text column missing from emotion_metadata."
)

goe_metadata = emotion_metadata[
    emotion_metadata["dataset"].astype(str).str.strip().eq("GoEmotions")
].copy()

assert len(goe_metadata) == 211225, (
    f"Unexpected GoEmotions metadata rows: {len(goe_metadata)}"
)

goe_clean_metadata = goe_metadata[
    goe_metadata["status"].eq("clean")
].copy()

assert len(goe_clean_metadata) == 133688, (
    f"Unexpected GoEmotions clean count: {len(goe_clean_metadata)}"
)

assert (
    goe_clean_metadata["text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
).all(), (
    "METADATA GATE FAIL — empty GoEmotions text exists in clean metadata."
)

print("✓ GoEmotions actual text payload preserved in emotion_metadata.")

valid_hash=emotion_metadata["duplicate_hash"].fillna("").ne("")
global_hash_counts=emotion_metadata.loc[valid_hash,"duplicate_hash"].value_counts()
emotion_metadata["global_duplicate_group_size"]=emotion_metadata["duplicate_hash"].map(global_hash_counts).fillna(0).astype(int)
emotion_metadata["global_duplicate_group_id"]=emotion_metadata["duplicate_hash"].where(valid_hash,emotion_metadata["sample_id"])

emotion_metadata["is_cross_dataset_duplicate"]=False
for digest, group in emotion_metadata.groupby("duplicate_hash",dropna=False):
    if digest and len(group)>1 and group["dataset"].nunique()>1:
        emotion_metadata.loc[group.index,"is_cross_dataset_duplicate"]=True

emotion_metadata.to_csv(CLEANED_DIR/"emotion_metadata_all.csv",index=False)

print("\n✓ Complete seven-class metadata construction finished.")

Processing CREMA-D...
  ✓ rows=7,442 clean=7,442 excluded=0
Processing RAVDESS...
  ✓ rows=2,880 clean=2,880 excluded=0
Processing SAVEE...
  ✓ rows=480 clean=480 excluded=0
Processing TESS...
  ✓ rows=2,800 clean=2,798 excluded=2
Processing IEMOCAP...
  ✓ rows=10,190 clean=5,680 excluded=4,510
Processing FER2013...
  ✓ rows=35,887 clean=35,887 excluded=0
Processing AffectNet...
  ✓ rows=30,626 clean=27,755 excluded=2,871
Processing CK+...
  ✓ rows=1,962 clean=1,854 excluded=108
Processing RAF-DB...
  ✓ rows=15,339 clean=15,339 excluded=0
Processing GoEmotions...

GoEmotions parsing audit:


,category,label,count
0,reason,ambiguous_multiple_canonical_classes,6507
1,reason,example_very_unclear,3411
2,reason,unsupported_only_labels,67619
3,reason,valid_one_canonical_class,133688
4,canonical_emotion,anger,26287
5,canonical_emotion,disgust,3438
6,canonical_emotion,fear,3492
7,canonical_emotion,happiness,26776
8,canonical_emotion,neutral,55298
9,canonical_emotion,sadness,13798


  ✓ rows=211,225 clean=133,688 excluded=77,537
Processing SAMM...
  ✓ rows=1 clean=0 excluded=1
✓ GoEmotions actual text payload preserved in emotion_metadata.

✓ Complete seven-class metadata construction finished.


In [9]:
# ============================================================
# HARD GOEMOTIONS QA — MUST PASS BEFORE FINAL MANIFEST
# ============================================================

GOE_ROOT = DATASET_PATHS["GoEmotions"]["path"]

goe_files = sorted(Path(GOE_ROOT).glob("*.csv"))

assert len(goe_files) == 3, (
    f"Expected 3 GoEmotions CSV files, found {len(goe_files)}"
)

raw_total = 0
reason_counts = {}
canonical_counts = {}
schema_checked = False

for f in goe_files:
    df_goe = pd.read_csv(f)
    raw_total += len(df_goe)

    cols = detect_goemotions_onehot_columns(df_goe)

    assert len(cols) == 28, (
        f"{f.name}: expected 28 one-hot columns, found {len(cols)}"
    )

    assert "text" in df_goe.columns, (
        f"{f.name}: missing text column"
    )

    schema_checked = True

    for _, row in df_goe.iterrows():
        label, reason, active, candidates = decode_goemotions_row(
            row, cols
        )

        reason_counts[reason] = reason_counts.get(reason, 0) + 1

        if label is not None:
            canonical_counts[label] = (
                canonical_counts.get(label, 0) + 1
            )

print("GoEmotions raw rows:", raw_total)

print(
    "GoEmotions schema: 28 one-hot labels PASS"
    if schema_checked
    else
    "FAIL"
)

print(
    "GoEmotions independent decoder reasons:",
    reason_counts
)

print(
    "GoEmotions independently valid canonical rows:",
    sum(canonical_counts.values())
)

print(
    "GoEmotions canonical distribution:",
    canonical_counts
)

assert raw_total == 211225, (
    f"Unexpected GoEmotions raw row count: {raw_total}"
)

assert schema_checked

assert sum(canonical_counts.values()) > 0

# Critical regression guard:
# old erroneous result = 17,131
old_bug_count = 17131

if sum(canonical_counts.values()) == old_bug_count:
    raise AssertionError(
        "REGRESSION: independent valid count reproduced "
        "the old erroneous 17,131-row GoEmotions result."
    )
else:
    print(
        "GoEmotions one-hot decoding regression guard: PASS"
    )
# =============================================================

GoEmotions raw rows: 211225
GoEmotions schema: 28 one-hot labels PASS
GoEmotions independent decoder reasons: {'valid_one_canonical_class': 133688, 'example_very_unclear': 3411, 'unsupported_only_labels': 67619, 'ambiguous_multiple_canonical_classes': 6507}
GoEmotions independently valid canonical rows: 133688
GoEmotions canonical distribution: {'sadness': 13798, 'neutral': 55298, 'happiness': 26776, 'anger': 26287, 'surprise': 4599, 'disgust': 3438, 'fear': 3492}
GoEmotions one-hot decoding regression guard: PASS


In [10]:
# CELL 8 — FINAL QA + FROZEN EMOTION MANIFEST
# =============================================================

# -------------------------------------------------------------
# 1. Keep only clean canonical emotion samples
# -------------------------------------------------------------

clean = emotion_metadata[
    emotion_metadata["status"].eq("clean")
    & emotion_metadata["mapped_emotion"].isin(CANONICAL_EMOTION_SET)
].copy()

if clean.empty:
    raise RuntimeError("No clean canonical samples were produced.")


# -------------------------------------------------------------
# 2. Preserve GoEmotions text payload BEFORE manifest freezing
# -------------------------------------------------------------
#
# GoEmotions is a TEXT dataset.
#
# Its parser creates the actual "text" field. The previous version
# accidentally removed that field in manifest_columns, leaving only
# modality="text" without the actual text payload.
#
# This is a hard requirement for downstream text preprocessing.
# -------------------------------------------------------------

if "GoEmotions" in set(clean["dataset"].astype(str)):

    goe_clean = clean[
        clean["dataset"].astype(str).str.strip().eq("GoEmotions")
    ]

    if "text" not in clean.columns:
        raise RuntimeError(
            "FINAL MANIFEST BUILD FAIL — GoEmotions text column "
            "was lost before manifest construction."
        )

    goe_text = (
        goe_clean["text"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    if goe_text.eq("").any():
        empty_count = int(goe_text.eq("").sum())
        raise RuntimeError(
            f"FINAL MANIFEST BUILD FAIL — {empty_count:,} "
            "GoEmotions rows have empty text."
        )

    print(
        f"✓ GoEmotions text payload verified before manifest freeze: "
        f"{len(goe_clean):,} rows"
    )


# -------------------------------------------------------------
# 3. Deterministic global duplicate ranking
# -------------------------------------------------------------

clean = clean.sort_values(
    [
        "global_duplicate_group_id",
        "dataset",
        "file_path",
        "sample_id",
    ]
).reset_index(drop=True)

clean["global_duplicate_rank"] = clean.groupby(
    "global_duplicate_group_id",
    dropna=False
).cumcount()


# -------------------------------------------------------------
# 4. Keep the first representative of every global duplicate
# -------------------------------------------------------------

training_manifest = clean[
    clean["global_duplicate_rank"].eq(0)
].copy()


# -------------------------------------------------------------
# 5. Canonical emotion IDs
# -------------------------------------------------------------

emotion_to_id = {
    emotion: idx
    for idx, emotion in enumerate(CANONICAL_EMOTIONS)
}

training_manifest["emotion_id"] = (
    training_manifest["mapped_emotion"]
    .map(emotion_to_id)
    .astype("int64")
)


# -------------------------------------------------------------
# 6. Split / duplicate identifiers
# -------------------------------------------------------------

training_manifest["split_group_id"] = (
    training_manifest["group_id"].astype(str)
)

training_manifest["duplicate_group_id"] = (
    training_manifest["global_duplicate_group_id"].astype(str)
)

training_manifest["split"] = "UNASSIGNED"


# -------------------------------------------------------------
# 7. FINAL MANIFEST COLUMNS
# -------------------------------------------------------------
#
# IMPORTANT:
# "text" is explicitly retained.
#
# It is populated for GoEmotions and remains NaN/empty for
# non-text datasets.
# -------------------------------------------------------------

manifest_columns = [
    "sample_id",
    "dataset",
    "task",
    "modality",
    "file_path",

    # Source annotation
    "original_label",
    "mapped_emotion",
    "emotion_id",

    # Actual GoEmotions text payload
    "text",

    # Group / split information
    "group_id",
    "group_source",
    "split_group_id",

    # Duplicate information
    "duplicate_hash",
    "duplicate_group_id",
    "duplicate_group_size",
    "global_duplicate_group_size",
    "global_duplicate_rank",
    "is_exact_duplicate",
    "is_cross_dataset_duplicate",

    # Split
    "split",
]


# -------------------------------------------------------------
# 8. Ensure every required column exists
# -------------------------------------------------------------

missing_manifest_columns = [
    col
    for col in manifest_columns
    if col not in training_manifest.columns
]

if missing_manifest_columns:
    raise RuntimeError(
        "FINAL MANIFEST BUILD FAIL — missing required columns: "
        f"{missing_manifest_columns}"
    )


# -------------------------------------------------------------
# 9. Freeze final manifest
# -------------------------------------------------------------

training_manifest = (
    training_manifest[manifest_columns]
    .sort_values(
        [
            "dataset",
            "mapped_emotion",
            "file_path",
            "sample_id",
        ]
    )
    .reset_index(drop=True)
)


# -------------------------------------------------------------
# 10. Final GoEmotions payload validation AFTER freezing
# -------------------------------------------------------------

goe_final = training_manifest[
    training_manifest["dataset"]
    .astype(str)
    .str.strip()
    .eq("GoEmotions")
].copy()

if len(goe_final) == 0:
    raise RuntimeError(
        "FINAL MANIFEST BUILD FAIL — GoEmotions absent after freeze."
    )

goe_text_final = (
    goe_final["text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

if goe_text_final.eq("").any():

    empty_count = int(goe_text_final.eq("").sum())

    raise RuntimeError(
        "FINAL MANIFEST BUILD FAIL — "
        f"{empty_count:,} GoEmotions rows lost/contain empty text "
        "after manifest freeze."
    )

print(
    f"✓ GoEmotions text retained in frozen manifest: "
    f"{len(goe_final):,} rows"
)


# -------------------------------------------------------------
# 11. Save final emotion training manifest
# -------------------------------------------------------------

manifest_path = (
    CLEANED_DIR /
    "final_emotion_training_manifest.csv"
)

training_manifest.to_csv(
    manifest_path,
    index=False
)


# -------------------------------------------------------------
# 12. Final distribution
# -------------------------------------------------------------

distribution = (
    training_manifest["mapped_emotion"]
    .value_counts()
    .reindex(
        CANONICAL_EMOTIONS,
        fill_value=0
    )
)


# -------------------------------------------------------------
# 13. QA report
# -------------------------------------------------------------

qa = {
    "metadata_rows_all":
        int(len(emotion_metadata)),

    "canonical_clean_before_global_dedup":
        int(len(clean)),

    "final_training_manifest_rows":
        int(len(training_manifest)),

    "excluded_rows":
        int(
            (
                emotion_metadata["status"]
                == "excluded"
            ).sum()
        ),

    "global_exact_duplicate_hash_groups":
        int(
            (global_hash_counts > 1).sum()
        ),

    "cross_dataset_duplicate_rows":
        int(
            emotion_metadata[
                "is_cross_dataset_duplicate"
            ].sum()
        ),

    "datasets_in_manifest":
        sorted(
            training_manifest[
                "dataset"
            ].unique().tolist()
        ),

    "modalities_in_manifest":
        sorted(
            training_manifest[
                "modality"
            ].unique().tolist()
        ),

    "canonical_emotions":
        CANONICAL_EMOTIONS,

    "emotion_to_id":
        emotion_to_id,

    "goemotions_included":
        True,

    "goemotions_text_column":
        "text",

    "goemotions_final_rows":
        int(len(goe_final)),

    "samm_included":
        False,

    "sarcasm_separate":
        True,

    "split_status":
        "UNASSIGNED_BY_DESIGN",
}


with open(
    CLEANED_DIR /
    "metadata_qa_report.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        qa,
        f,
        indent=4
    )


distribution.to_csv(
    CLEANED_DIR /
    "final_emotion_distribution.csv",
    header=True
)


# -------------------------------------------------------------
# 14. FINAL CLEANED EMOTION METADATA QA
# -------------------------------------------------------------

print(
    "\n" + "=" * 90
)

print(
    "FINAL CLEANED EMOTION METADATA QA"
)

print(
    "=" * 90
)

print(
    f"All metadata rows: "
    f"{len(emotion_metadata):,}"
)

print(
    f"Canonical clean before global dedup: "
    f"{len(clean):,}"
)

print(
    f"Final training manifest rows: "
    f"{len(training_manifest):,}"
)

print(
    f"Excluded rows: "
    f"{qa['excluded_rows']:,}"
)

print(
    f"Cross-dataset duplicate rows: "
    f"{qa['cross_dataset_duplicate_rows']:,}"
)


print(
    "\nFinal canonical distribution:"
)

for emotion in CANONICAL_EMOTIONS:

    print(
        f"  {emotion:10s}: "
        f"{distribution[emotion]:,}"
    )


print(
    "\nDatasets in final manifest:"
)

for dataset in qa["datasets_in_manifest"]:

    print(
        f"  ✓ {dataset}"
    )# -------------------------------------------------------------
# FINAL HARD GATES
# -------------------------------------------------------------

required_datasets = {
    "CREMA-D","RAVDESS","SAVEE","TESS","IEMOCAP",
    "FER2013","AffectNet","CK+","RAF-DB","GoEmotions"
}

present = set(
    training_manifest["dataset"].astype(str).str.strip().unique()
)

missing = required_datasets - present

zero_classes = [
    e for e in CANONICAL_EMOTIONS
    if int(distribution[e]) == 0
]

if missing:
    raise RuntimeError(
        "FINAL GATE FAIL — required datasets absent from manifest: "
        f"{sorted(missing)}"
    )

if zero_classes:
    raise RuntimeError(
        "FINAL GATE FAIL — canonical classes absent: "
        f"{zero_classes}"
    )

# The actual text payload must exist, not merely modality="text".
if "text" not in training_manifest.columns:
    raise RuntimeError(
        "FINAL GATE FAIL — actual text payload column is absent."
    )

if "GoEmotions" not in present:
    raise RuntimeError(
        "FINAL GATE FAIL — GoEmotions absent from final manifest."
    )

goe_final = training_manifest[
    training_manifest["dataset"].astype(str).str.strip().eq("GoEmotions")
].copy()

assert len(goe_final) > 0, (
    "FINAL GATE FAIL — GoEmotions absent from final manifest."
)

assert (
    goe_final["modality"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("text")
).all(), (
    "FINAL GATE FAIL — GoEmotions modality is not text."
)

assert goe_final["mapped_emotion"].isin(
    CANONICAL_EMOTION_SET
).all(), (
    "FINAL GATE FAIL — non-canonical GoEmotions labels leaked."
)

goe_text_final = (
    goe_final["text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

assert goe_text_final.ne("").all(), (
    "FINAL GATE FAIL — empty GoEmotions text leaked into final manifest."
)

dataset_clean_counts = (
    training_manifest.groupby("dataset").size().to_dict()
)

zero_clean_datasets = sorted(
    d for d in required_datasets
    if int(dataset_clean_counts.get(d, 0)) == 0
)

if zero_clean_datasets:
    raise RuntimeError(
        "FINAL GATE FAIL — required datasets have zero clean samples: "
        f"{zero_clean_datasets}"
    )

print("\nFINAL DATASET GATE: PASS")
print("All required emotion datasets are represented with clean canonical samples.")
print("GoEmotions text modality is present.")
print("GoEmotions actual text payload is preserved.")
print("All seven canonical emotion classes are represented.")
print("Split status: UNASSIGNED (splitting is performed in the dedicated split notebook).")

print("\nFINAL MANIFEST:")
print(f"  Rows: {len(training_manifest):,}")
print(f"  GoEmotions rows: {len(goe_final):,}")
print(f"  Columns: {len(training_manifest.columns)}")
print(f"  Saved: {manifest_path}")

✓ GoEmotions text payload verified before manifest freeze: 133,688 rows
✓ GoEmotions text retained in frozen manifest: 50,831 rows

FINAL CLEANED EMOTION METADATA QA
All metadata rows: 318,832
Canonical clean before global dedup: 233,803
Final training manifest rows: 146,335
Excluded rows: 85,029
Cross-dataset duplicate rows: 0

Final canonical distribution:
  anger     : 21,411
  disgust   : 7,238
  fear      : 11,764
  happiness : 34,249
  sadness   : 19,523
  surprise  : 11,899
  neutral   : 40,251

Datasets in final manifest:
  ✓ AffectNet
  ✓ CK+
  ✓ CREMA-D
  ✓ FER2013
  ✓ GoEmotions
  ✓ IEMOCAP
  ✓ RAF-DB
  ✓ RAVDESS
  ✓ SAVEE
  ✓ TESS

FINAL DATASET GATE: PASS
All required emotion datasets are represented with clean canonical samples.
GoEmotions text modality is present.
GoEmotions actual text payload is preserved.
All seven canonical emotion classes are represented.
Split status: UNASSIGNED (splitting is performed in the dedicated split notebook).

FINAL MANIFEST:
  Rows: 146,